# LeetCode 702 — Search in a Sorted Array of Unknown Size bằng Python

Link đề: https://leetcode.com/problems/search-in-a-sorted-array-of-unknown-size/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Interactive

Skeleton LeetCode:

```python
# """
# This is ArrayReader's API interface.
# You should not implement it, or speculate about its implementation
# """
#class ArrayReader:
#    def get(self, index: int) -> int:

class Solution:
    def search(self, reader: 'ArrayReader', target: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium** và là bài **tương tác (interactive)**. Đề trong notebook được viết theo hiểu biết về bài, nên nếu bạn có tài khoản Premium hãy đối chiếu lại đề gốc (đặc biệt ví dụ và ràng buộc). Vì trên máy ta không có hệ chấm của LeetCode, notebook **tự dựng lớp `ArrayReader`** mô phỏng cách LeetCode gọi: lớp này giữ mảng bí mật, trả về `secret[i]` hoặc `2^31 - 1` nếu `i` ngoài mảng, và **đếm số lần gọi `get`** để ta kiểm tra yêu cầu `O(log n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có một mảng bí mật đã **sắp xếp tăng chặt** nhưng **không biết độ dài**; ta chỉ được hỏi từng phần tử qua `ArrayReader.get(i)` (trả `2^31 - 1` nếu `i` vượt biên). Hãy tìm chỉ số của `target` hoặc trả `-1`, với độ phức tạp **`O(log n)`** số lần gọi.
2. Nắm các công cụ Python: viết một **lớp** (`class`) với phương thức, bộ đếm số lần gọi, tìm kiếm nhị phân bằng `while lo <= hi`, nhân đôi `r *= 2`, `bisect` với đối tượng giống dãy (`__getitem__`, `__len__`).
3. Đi từ **duyệt tuyến tính** → **nhân đôi (exponential/galloping) để tìm cận phải rồi tìm nhị phân** → **dùng cận cố định theo ràng buộc `10^4` (hoặc `2^31 - 1`)** → **tìm độ dài mảng trước rồi tìm nhị phân thường**.
4. Hiểu mấu chốt: giá trị trả về khi **ngoài biên** là `2^31 - 1` — **lớn hơn mọi giá trị hợp lệ** nên "ngoài biên" tự nhiên đóng vai trò `+vô cùng` và **không phá tính đơn điệu** của dãy; nhờ đó ta có thể tìm nhị phân ngay cả khi hỏi quá biên.
5. Tránh các lỗi kinh điển: khởi tạo `r = 0` rồi nhân đôi (mãi bằng 0, lặp vô hạn), bỏ sót phần tử ở chỗ dừng khi đặt `hi = r - 1`, duyệt tuyến tính bị quá giới hạn gọi `get`, quên rằng chỉ số âm không hợp lệ, và nhầm đối tượng `reader` với mảng thường (dùng `len(reader)` hay `reader[i]`).

## 2. Đề bài & ví dụ

Có một mảng **đã sắp xếp tăng chặt (các phần tử khác nhau)** với **kích thước không biết trước**. Bạn **không** truy cập trực tiếp được mảng, chỉ dùng được giao diện `ArrayReader`:

- `ArrayReader.get(i)` trả về giá trị tại chỉ số `i` (đánh số từ 0) của mảng bí mật `secret[i]`, **hoặc** trả về `2^31 - 1` nếu `i` **nằm ngoài biên** mảng.

Cho số nguyên `target`. Trả về chỉ số `k` sao cho `secret[k] == target`, nếu không có thì trả về `-1`.

Phải viết thuật toán có độ phức tạp **`O(log n)`** (số lần gọi `get`).

**Ví dụ 1**

- Input: `secret = [-1,0,3,5,9,12]`, `target = 9`
- Output: `4`
- Giải thích: `9` có trong mảng, ở chỉ số `4`.

**Ví dụ 2**

- Input: `secret = [-1,0,3,5,9,12]`, `target = 2`
- Output: `-1`
- Giải thích: `2` không có trong mảng.

**Ràng buộc:**

- `1 <= secret.length <= 10^4`
- `-10^4 <= secret[i], target <= 10^4`
- `secret` được sắp xếp theo thứ tự **tăng chặt**.

Chú ý: `target` và mọi phần tử đều nhỏ hơn `2^31 - 1`, nên giá trị "ngoài biên" luôn **lớn hơn** `target`.

In [1]:
from typing import List
from bisect import bisect_left
import math
import random
import time

INF = 2**31 - 1


class ArrayReader:
    """Mô phỏng ArrayReader của LeetCode (tự dựng cho notebook).

    - get(i): trả về secret[i], hoặc 2**31 - 1 nếu i vượt biên (i >= len).
    - đếm số lần gọi để kiểm tra yêu cầu O(log n).
    - chỉ số âm không hợp lệ -> báo lỗi (để bắt bug)."""

    def __init__(self, secret):
        self._secret = list(secret)
        self.calls = 0

    def get(self, index: int) -> int:
        self.calls += 1
        if index < 0:
            raise IndexError(f"chỉ số âm không hợp lệ: {index}")
        if index >= len(self._secret):
            return INF
        return self._secret[index]


examples = [
    ([-1, 0, 3, 5, 9, 12], 9, 4),
    ([-1, 0, 3, 5, 9, 12], 2, -1),
]
r = ArrayReader([-1, 0, 3, 5, 9, 12])
print("get(0) =", r.get(0), "| get(4) =", r.get(4), "| get(5) =", r.get(5), "| get(6) =", r.get(6), "| get(1000) =", r.get(1000))
print("đã gọi get", r.calls, "lần")
for secret, target, expected in examples:
    print(f"secret = {secret}, target = {target} -> kỳ vọng {expected}")

get(0) = -1 | get(4) = 9 | get(5) = 12 | get(6) = 2147483647 | get(1000) = 2147483647
đã gọi get 5 lần
secret = [-1, 0, 3, 5, 9, 12], target = 9 -> kỳ vọng 4
secret = [-1, 0, 3, 5, 9, 12], target = 2 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Lớp (`class`) và đối tượng có trạng thái

`ArrayReader` ở trên là một **lớp**: `__init__` khởi tạo thuộc tính (`_secret`, `calls`), `get` là **phương thức** (có thể thay đổi trạng thái, ở đây tăng bộ đếm). Mỗi đối tượng `ArrayReader(secret)` là một "máy hỏi đáp" riêng; ta không được (và không nên) nhìn vào `_secret` từ lời giải — chỉ gọi `get`. Bộ đếm `calls` giúp ta **đo** số lần hỏi.

In [2]:
r = ArrayReader([10, 20, 30])
before = r.calls
r.get(1)
r.get(7)
print("số lần gọi thêm:", r.calls - before)
try:
    r.get(-1)
except IndexError as ex:
    print("lỗi như mong đợi:", ex)

số lần gọi thêm: 2
lỗi như mong đợi: chỉ số âm không hợp lệ: -1


### 3.2. Tìm kiếm nhị phân cơ bản

Nếu mảng đã sắp xếp **và biết độ dài**, tìm nhị phân loại một nửa mỗi bước: giữ khoảng `[lo, hi]`, lấy `mid = (lo + hi) // 2`; nếu `a[mid] == target` thì xong; nếu `a[mid] < target` thì tìm bên phải (`lo = mid + 1`); ngược lại bên trái (`hi = mid - 1`). Số bước `O(log n)`.

In [3]:
def binary_search(a, target):
    lo, hi = 0, len(a) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if a[mid] == target:
            return mid, steps
        if a[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1, steps

a = [-1, 0, 3, 5, 9, 12]
print(binary_search(a, 9), binary_search(a, 2), binary_search(a, -1), binary_search(a, 12))
big = list(range(0, 20000, 2))
print("10^4 phần tử:", binary_search(big, 12344), "(chỉ ~14 bước)")

(4, 2) (-1, 3) (0, 2) (5, 3)
10^4 phần tử: (6172, 12) (chỉ ~14 bước)


### 3.3. Khó khăn: không biết độ dài

Bước đầu của tìm nhị phân là đặt `hi = len(a) - 1`. Ở bài này **không có `len`**. Nếu ta cứ đoán một `hi` quá lớn thì sao? Nhờ quy ước "ngoài biên trả `2^31 - 1`" — một số **lớn hơn mọi `target` hợp lệ** — dãy mở rộng `secret[0], ..., secret[n-1], INF, INF, INF, ...` vẫn **tăng (không giảm)**. Tìm nhị phân trên dãy mở rộng này vẫn đúng: mỗi lần hỏi quá biên, ta nhận `INF > target` và hiểu rằng "phải tìm bên trái".

In [4]:
secret = [-1, 0, 3, 5, 9, 12]
extended = secret + [INF] * 5
print("dãy mở rộng:", extended)
print("không giảm?", all(extended[i] <= extended[i + 1] for i in range(len(extended) - 1)))
print("mọi phần tử thật < INF?", all(x < INF for x in secret))

dãy mở rộng: [-1, 0, 3, 5, 9, 12, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647]
không giảm? True
mọi phần tử thật < INF? True


### 3.4. Nhân đôi để tìm cận (exponential search / galloping)

Để **tìm một cận phải** `r` mà ta biết chắc `secret[r] >= target` (hoặc `r` đã ngoài biên), ta nhân đôi `r` từ `1`: `1, 2, 4, 8, ...` cho đến khi `get(r) >= target`. Vì `r` tăng gấp đôi mỗi bước nên chỉ cần `O(log n)` bước, và `r` cuối cùng **không quá `2n`** (vì `r = 2^k` đầu tiên thoả `2^k >= n` thì `2^k < 2n`, mà tại đó hoặc đã ngoài biên hoặc chạm giá trị `>= target`).

In [5]:
secret = list(range(0, 2000, 2))       # n = 1000
reader = ArrayReader(secret)
target = 1500
r = 1
while reader.get(r) < target:
    print(f"get({r}) = {reader.get(r)} < {target} -> nhân đôi")
    r *= 2
print(f"dừng ở r = {r}: get({r}) = {reader.get(r)} >= {target}")
print("n = 1000, 2n = 2000: r =", r, "<= 2n:", r <= 2 * len(secret))

get(1) = 2 < 1500 -> nhân đôi
get(2) = 4 < 1500 -> nhân đôi
get(4) = 8 < 1500 -> nhân đôi
get(8) = 16 < 1500 -> nhân đôi
get(16) = 32 < 1500 -> nhân đôi
get(32) = 64 < 1500 -> nhân đôi
get(64) = 128 < 1500 -> nhân đôi
get(128) = 256 < 1500 -> nhân đôi
get(256) = 512 < 1500 -> nhân đôi
get(512) = 1024 < 1500 -> nhân đôi
dừng ở r = 1024: get(1024) = 2147483647 >= 1500
n = 1000, 2n = 2000: r = 1024 <= 2n: True


### 3.5. Đối tượng giống dãy và `bisect`

`bisect.bisect_left(seq, x)` chạy tìm nhị phân trên mọi đối tượng có `__len__` và `__getitem__`. Ta có thể bọc `reader` vào một lớp giả như vậy để dùng thư viện (xem cách 3 ở mục 7). Ví dụ minh họa với một lớp bọc đơn giản:

In [6]:
class FakeSeq:
    def __init__(self, data):
        self.data = data
        self.accesses = 0
    def __len__(self):
        return 10**4 + 1
    def __getitem__(self, i):
        self.accesses += 1
        return self.data[i] if i < len(self.data) else INF

s = FakeSeq([-1, 0, 3, 5, 9, 12])
print("bisect_left cho 9:", bisect_left(s, 9), "| số lần truy cập:", s.accesses)

bisect_left cho 9: 4 | số lần truy cập: 13


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Lấy `secret = [-1, 0, 3, 5, 9, 12]` (ta biết nhưng thuật toán thì **không**) và `target = 9`.

Ta thử "chụp cận phải" bằng cách hỏi các chỉ số `1, 2, 4, 8, ...`:

| hỏi | `get` | so với `target = 9` |
|---|---|---|
| `get(1)` | `0` | `0 < 9`: còn nhỏ, tiến tiếp |
| `get(2)` | `3` | `3 < 9`: tiến tiếp |
| `get(4)` | `9` | `9 >= 9`: dừng! |

Biết `get(2) = 3 < 9` và `get(4) = 9 >= 9` nghĩa là `target` (nếu có) nằm trong chỉ số `[2, 4]`. Giờ tìm nhị phân trong đoạn `[2, 4]`: `mid = 3`, `get(3) = 5 < 9` → `[4, 4]`; `get(4) = 9` → tìm thấy chỉ số `4` ✓.

Với `target = 2` (không có): `get(1) = 0 < 2`, `get(2) = 3 >= 2` dừng, đoạn `[1, 2]`; `get(1) = 0 < 2` → `lo = 2`; `get(2) = 3 > 2` → `hi = 1`; kết thúc, trả `-1`.

**Nhận xét:** nhân đôi cận phải giúp ta **bao** `target` trong một đoạn có độ dài tỉ lệ với vị trí cần tìm, rồi tìm nhị phân trong đoạn đó. Tổng số lần hỏi gần `2 * log2(vị trí)`.

In [7]:
secret = [-1, 0, 3, 5, 9, 12]
for target in (9, 2):
    reader = ArrayReader(secret)
    r = 1
    while reader.get(r) < target:
        r *= 2
    lo, hi = r // 2, r
    print(f"target = {target}: cận phải r = {r} -> tìm nhị phân trong [{lo}, {hi}] | đã gọi {reader.calls} lần")

target = 9: cận phải r = 4 -> tìm nhị phân trong [2, 4] | đã gọi 3 lần
target = 2: cận phải r = 2 -> tìm nhị phân trong [1, 2] | đã gọi 2 lần


## 5. Cách 1: Brute Force (duyệt tuyến tính)

### Ý tưởng

Hỏi lần lượt `get(0), get(1), get(2), ...`. Gặp `target` thì trả chỉ số. Dừng khi gặp `2^31 - 1` (hết mảng) và trả `-1`. (Có thể dừng sớm khi gặp giá trị `> target` vì mảng tăng.)

In [8]:
def search_linear(reader, target):
    i = 0
    while True:
        v = reader.get(i)
        if v == target:
            return i
        if v > target:          # mảng tăng chặt: sau đó chỉ lớn hơn; INF cũng rơi vào nhánh này
            return -1
        i += 1


for secret, target, expected in examples:
    reader = ArrayReader(secret)
    got = search_linear(reader, target)
    print(f"secret = {secret}, target = {target} -> {got} (kỳ vọng {expected}) | số lần gọi get: {reader.calls}")
    assert got == expected

secret = [-1, 0, 3, 5, 9, 12], target = 9 -> 4 (kỳ vọng 4) | số lần gọi get: 5
secret = [-1, 0, 3, 5, 9, 12], target = 2 -> -1 (kỳ vọng -1) | số lần gọi get: 3


### Độ phức tạp

- Số lần gọi `get` là `O(n)` (tới vị trí của `target`, tối đa `n + 1`). Đề yêu cầu **`O(log n)`** nên cách này **không đạt** (dù với `n <= 10^4` vẫn chạy được, nhưng sẽ bị hệ chấm coi là không thỏa yêu cầu độ phức tạp/số lần gọi).
- Space `O(1)`.

### Nút thắt

Cách này **không tận dụng việc mảng đã sắp xếp** ngoài việc dừng sớm. Mỗi lần hỏi chỉ loại bỏ **một** phần tử, trong khi tìm nhị phân loại **một nửa**. Hãy đo số lần hỏi:

In [9]:
for n in (100, 1000, 10000):
    secret = list(range(0, 2 * n, 2))
    reader = ArrayReader(secret)
    search_linear(reader, secret[-1])               # tìm phần tử cuối: trường hợp xấu
    print(f"n = {n:>5}: duyệt tuyến tính gọi get {reader.calls:>6} lần | log2(n) ≈ {math.log2(n):5.1f}")

n =   100: duyệt tuyến tính gọi get    100 lần | log2(n) ≈   6.6
n =  1000: duyệt tuyến tính gọi get   1000 lần | log2(n) ≈  10.0
n = 10000: duyệt tuyến tính gọi get  10000 lần | log2(n) ≈  13.3


Số lần hỏi tỉ lệ thuận với `n` (so với `~14` khi `n = 10^4` nếu tìm nhị phân). Cần cách tận dụng thứ tự để mỗi lần hỏi loại cả một nửa — nhưng ta lại **chưa biết `n`**.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Ngoài biên trả `2^31 - 1`, lớn hơn mọi `target`.** Do đó, nếu ta hỏi một chỉ số `r` quá lớn, ta nhận được `INF > target` — một thông tin "đúng hướng" (cần tìm sang **trái**), không làm hỏng việc tìm nhị phân.
2. **Ta không cần biết `n` chính xác**, chỉ cần một **cận phải** `r` với `get(r) >= target` (có thể ngoài biên). Tìm cận bằng cách **nhân đôi** `r = 1, 2, 4, ...` mất `O(log n)` lần hỏi; `r` không vượt quá `2n`, nên tìm nhị phân trong `[r/2, r]` cũng mất `O(log n)` lần.

Hợp lại: `O(log n)` lần hỏi, thoả đề. Có các biến thể của cùng tinh thần:

- **Cách 2:** nhân đôi tìm cận → tìm nhị phân trong `[r/2, r]` (cách nộp).
- **Cách 3:** dùng **cận cố định** (theo ràng buộc `n <= 10^4`, hay thậm chí `2^31 - 1`) rồi tìm nhị phân thẳng, tận dụng `INF` ngoài biên.
- **Cách 4:** **tìm độ dài `n`** trước (tìm nhị phân điểm "hết mảng"), rồi tìm nhị phân thông thường trên `[0, n - 1]`.

In [10]:
# minh họa: số lần hỏi để chốt cận phải bằng nhân đôi ~ log2 của vị trí cần tìm
for n in (10, 100, 1000, 10000):
    secret = list(range(0, 2 * n, 2))
    reader = ArrayReader(secret)
    r = 1
    while reader.get(r) < secret[-1]:
        r *= 2
    print(f"n = {n:>5}: tìm phần tử cuối: r = {r:>6} (<= 2n = {2 * n}), số lần hỏi để chốt cận = {reader.calls}")

n =    10: tìm phần tử cuối: r =     16 (<= 2n = 20), số lần hỏi để chốt cận = 5
n =   100: tìm phần tử cuối: r =    128 (<= 2n = 200), số lần hỏi để chốt cận = 8
n =  1000: tìm phần tử cuối: r =   1024 (<= 2n = 2000), số lần hỏi để chốt cận = 11
n = 10000: tìm phần tử cuối: r =  16384 (<= 2n = 20000), số lần hỏi để chốt cận = 15


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Nhân đôi tìm cận rồi tìm nhị phân (nộp)

**Giải thích từng đoạn:**

1. Đặt `r = 1`. Trong khi `reader.get(r) < target`: `r *= 2`. Khi dừng, `get(r) >= target` (có thể `r` đã vượt biên và `get(r) = INF`).
2. Phần tử `target` (nếu tồn tại) nằm trong chỉ số `[r // 2, r]`: vì ở bước trước `get(r // 2) < target` (hoặc `r // 2 = 0`, tức `r = 1`, nơi ta chưa kiểm tra `get(0)`), nên chỉ số `< r // 2` đều nhỏ hơn `target`.
3. Tìm nhị phân chuẩn với `lo = r // 2`, `hi = r`; mỗi lần `get(mid)`; nếu trả `INF` (vượt biên) coi là `> target`.
4. Không thấy thì trả `-1`.

**Code demo (có in số lần gọi `get`):**

In [11]:
def search_doubling_demo(reader, target):
    r = 1
    while reader.get(r) < target:
        print(f"  get({r}) < {target} -> r = {2 * r}")
        r *= 2
    print(f"  get({r}) = {reader.get(r)} >= {target} -> dừng nhân đôi, tìm nhị phân trong [{r // 2}, {r}]")
    lo, hi = r // 2, r
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        print(f"  [{lo}, {hi}] mid = {mid}, get = {v if v != INF else 'INF'}")
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

for target in (9, 2, 12, -1, 100):
    reader = ArrayReader([-1, 0, 3, 5, 9, 12])
    print(f"target = {target}:")
    print("  ->", search_doubling_demo(reader, target), f"| tổng số lần gọi get: {reader.calls}")

target = 9:
  get(1) < 9 -> r = 2
  get(2) < 9 -> r = 4
  get(4) = 9 >= 9 -> dừng nhân đôi, tìm nhị phân trong [2, 4]
  [2, 4] mid = 3, get = 5
  [4, 4] mid = 4, get = 9
  -> 4 | tổng số lần gọi get: 6
target = 2:
  get(1) < 2 -> r = 2
  get(2) = 3 >= 2 -> dừng nhân đôi, tìm nhị phân trong [1, 2]
  [1, 2] mid = 1, get = 0
  [2, 2] mid = 2, get = 3
  -> -1 | tổng số lần gọi get: 5
target = 12:
  get(1) < 12 -> r = 2
  get(2) < 12 -> r = 4
  get(4) < 12 -> r = 8
  get(8) = 2147483647 >= 12 -> dừng nhân đôi, tìm nhị phân trong [4, 8]
  [4, 8] mid = 6, get = INF
  [4, 5] mid = 4, get = 9
  [5, 5] mid = 5, get = 12
  -> 5 | tổng số lần gọi get: 8
target = -1:
  get(1) = 0 >= -1 -> dừng nhân đôi, tìm nhị phân trong [0, 1]
  [0, 1] mid = 0, get = -1
  -> 0 | tổng số lần gọi get: 3
target = 100:
  get(1) < 100 -> r = 2
  get(2) < 100 -> r = 4
  get(4) < 100 -> r = 8
  get(8) = 2147483647 >= 100 -> dừng nhân đôi, tìm nhị phân trong [4, 8]
  [4, 8] mid = 6, get = INF
  [4, 5] mid = 4, get = 9
  

**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    def search(self, reader: "ArrayReader", target: int) -> int:
        r = 1
        while reader.get(r) < target:
            r *= 2
        lo, hi = r // 2, r
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid)
            if v == target:
                return mid
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1


for secret, target, expected in examples:
    assert Solution2().search(ArrayReader(secret), target) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** số lần gọi `get`: nhân đôi `O(log n)` + tìm nhị phân trên đoạn dài `r / 2 + 1 <= n + 1` nên `O(log n)` ⇒ tổng `O(log n)`. Space `O(1)`.

### 7.2. Cách 3 — Cận cố định rồi tìm nhị phân (tận dụng `INF`)

**Ý tưởng khác:** nếu ta **biết** `n <= 10^4` (ràng buộc của đề), ta khỏi cần nhân đôi: đặt `hi = 10^4` (hoặc `hi = 10^4 - 1`) và tìm nhị phân luôn. Mọi chỉ số `>= n` trả `INF > target` nên được hiểu "phải đi trái" — chạy đúng.

Ta thậm chí có thể không dựa vào ràng buộc về `n`: đặt `hi = 2^31 - 1` (số chỉ số lớn nhất có thể "hỏi"), tìm nhị phân tốn `~31` lần hỏi.

**Giải thích từng đoạn:**

1. `lo = 0`, `hi = 10**4` (hoặc `INF`).
2. Mỗi lần `mid = (lo + hi) // 2`, `v = reader.get(mid)`; so sánh với `target` như tìm nhị phân thường.
3. Nếu `v == target` trả `mid`; nếu `v < target` thì `lo = mid + 1`; ngược lại (kể cả `v = INF`) thì `hi = mid - 1`.

**Code demo (cận `10^4`) và so sánh với cận `2^31 - 1`:**

In [13]:
def search_fixed_bound(reader, target, hi):
    lo = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


for secret, target, expected in examples:
    for hi in (10**4, INF):
        reader = ArrayReader(secret)
        got = search_fixed_bound(reader, target, hi)
        print(f"hi = {hi:>10}: target = {target} -> {got} (kỳ vọng {expected}), số lần gọi get = {reader.calls}")
        assert got == expected

hi =      10000: target = 9 -> 4 (kỳ vọng 4), số lần gọi get = 13
hi = 2147483647: target = 9 -> 4 (kỳ vọng 4), số lần gọi get = 31
hi =      10000: target = 2 -> -1 (kỳ vọng -1), số lần gọi get = 13
hi = 2147483647: target = 2 -> -1 (kỳ vọng -1), số lần gọi get = 31


**Code nộp LeetCode (cách 3, cận `10^4` theo ràng buộc):**

In [14]:
class Solution3:
    def search(self, reader: "ArrayReader", target: int) -> int:
        lo, hi = 0, 10**4
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid)
            if v == target:
                return mid
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1


for secret, target, expected in examples:
    assert Solution3().search(ArrayReader(secret), target) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** số lần gọi `get` là `O(log H)` với `H` là cận (ở đây `H = 10^4` nên khoảng 14 lần; với `H = 2^31 - 1` khoảng 31 lần). Space `O(1)`. Ưu điểm: đơn giản; nhược điểm: **dựa vào ràng buộc** `n <= 10^4` (nếu bỏ ràng buộc, phải dùng cận rất lớn); số lần hỏi **luôn cỡ `log H`** dù mảng thật ngắn, trong khi nhân đôi chỉ tốn `O(log n)` theo `n` thật.

**Biến thể dùng thư viện:** bọc `reader` thành một dãy giả và gọi `bisect_left`:

In [15]:
class ReaderSeq:
    """Bọc ArrayReader thành đối tượng giống dãy có độ dài giả 10^4 + 1."""
    def __init__(self, reader, size=10**4 + 1):
        self.reader, self.size = reader, size
    def __len__(self):
        return self.size
    def __getitem__(self, i):
        return self.reader.get(i)


def search_bisect(reader, target):
    seq = ReaderSeq(reader)
    i = bisect_left(seq, target)            # chỉ số đầu tiên có giá trị >= target (INF tính là rất lớn)
    return i if reader.get(i) == target else -1

for secret, target, expected in examples:
    reader = ArrayReader(secret)
    got = search_bisect(reader, target)
    print(f"target = {target}: {got} (kỳ vọng {expected}), số lần gọi get = {reader.calls}")
    assert got == expected

target = 9: 4 (kỳ vọng 4), số lần gọi get = 14
target = 2: -1 (kỳ vọng -1), số lần gọi get = 14


### 7.3. Cách 4 — Tìm độ dài `n` trước, rồi tìm nhị phân thường

**Ý tưởng khác:** tách thành **hai bài toán** quen thuộc:

1. **Tìm độ dài `n`** của mảng: `n` là chỉ số nhỏ nhất `i` mà `get(i) == INF`. Dùng nhân đôi để chốt một cận `r` có `get(r) == INF`, rồi tìm nhị phân "chỉ số đầu tiên trả `INF`" trong `[r/2, r]`.
2. **Tìm nhị phân thông thường** trên `[0, n - 1]` đã biết độ dài.

**Giải thích từng đoạn:**

1. `r = 1`; trong khi `get(r) != INF`: `r *= 2`. Lúc dừng `get(r) == INF` nên `r >= n`; và `get(r // 2) != INF` (hoặc `r = 1`) nên `r // 2 < n` (khi `r > 1`).
2. Tìm nhị phân điểm chuyển: `lo = r // 2`, `hi = r`; tìm `n` = chỉ số nhỏ nhất có `get == INF`: nếu `get(mid) == INF` thì `hi = mid` (điểm chuyển ở `mid` hoặc trước nó) ngược lại `lo = mid + 1`.
3. Khi `lo == hi` đó là `n`. Tìm nhị phân thường trên `[0, n - 1]`.

**Code demo:**

In [16]:
def find_length(reader):
    r = 1
    while reader.get(r) != INF:
        r *= 2
    lo, hi = r // 2, r                      # n nằm trong [r//2, r] (n = chỉ số đầu tiên trả INF)
    while lo < hi:
        mid = (lo + hi) // 2
        if reader.get(mid) == INF:
            hi = mid
        else:
            lo = mid + 1
    return lo


def search_by_length_demo(reader, target):
    n = find_length(reader)
    print(f"  độ dài tìm được n = {n} sau {reader.calls} lần gọi get")
    lo, hi = 0, n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

for target in (9, 2):
    reader = ArrayReader([-1, 0, 3, 5, 9, 12])
    print(f"target = {target}:")
    print("  ->", search_by_length_demo(reader, target), f"| tổng số lần gọi get: {reader.calls}")

target = 9:
  độ dài tìm được n = 6 sau 6 lần gọi get
  -> 4 | tổng số lần gọi get: 8
target = 2:
  độ dài tìm được n = 6 sau 6 lần gọi get
  -> -1 | tổng số lần gọi get: 9


**Code nộp LeetCode (cách 4):**

In [17]:
class Solution4:
    def search(self, reader: "ArrayReader", target: int) -> int:
        INF_ = 2**31 - 1
        r = 1
        while reader.get(r) != INF_:
            r *= 2
        lo, hi = r // 2, r
        while lo < hi:                      # tìm độ dài n
            mid = (lo + hi) // 2
            if reader.get(mid) == INF_:
                hi = mid
            else:
                lo = mid + 1
        n = lo
        lo, hi = 0, n - 1                   # tìm nhị phân thường
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid)
            if v == target:
                return mid
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1


for secret, target, expected in examples:
    assert Solution4().search(ArrayReader(secret), target) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp:** số lần gọi `get` `O(log n)` cho pha tìm độ dài + `O(log n)` cho tìm nhị phân ⇒ `O(log n)`; hằng số lớn hơn cách 2 một chút (khoảng `3 log n`). Space `O(1)`. Lợi ích: **tách bài toán** rõ ràng và cho biết độ dài mảng (hữu ích nếu cần dùng lại).

(Trường hợp `n = 1`, `r = 1`: `get(1) = INF` nên không nhân đôi; `lo = 0`, `hi = 1`, `mid = 0`, `get(0) != INF` nên `lo = 1` → `n = 1` ✓.)

## 8. TRACE từng bước

### 8.1. Trace cách 2 (nhân đôi rồi tìm nhị phân), `secret = [-1,0,3,5,9,12]`, `target = 9`

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `pha` | "nhân đôi" hoặc "nhị phân" |
| `r` hoặc `[lo, hi]` | cận hiện tại |
| `mid` / chỉ số hỏi | chỉ số gọi `get` |
| `get` | giá trị nhận được (`INF` nếu ngoài biên) |

In [18]:
def trace_doubling(secret, target):
    reader = ArrayReader(secret)
    print(f"secret (ẩn) = {secret}, target = {target}")
    r = 1
    while True:
        v = reader.get(r)
        shown = "INF" if v == INF else v
        print(f"[nhân đôi] r = {r:>3} | get({r}) = {shown:>3} | {'< target -> r *= 2' if v < target else '>= target -> dừng'}")
        if v >= target:
            break
        r *= 2
    lo, hi = r // 2, r
    print(f"[nhị phân] khoảng [{lo}, {hi}]")
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        shown = "INF" if v == INF else v
        if v == target:
            print(f"[nhị phân] lo = {lo}, hi = {hi}, mid = {mid} | get = {shown} == target -> TÌM THẤY")
            print("tổng số lần gọi get:", reader.calls)
            return mid
        if v < target:
            print(f"[nhị phân] lo = {lo}, hi = {hi}, mid = {mid} | get = {shown} < target -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"[nhị phân] lo = {lo}, hi = {hi}, mid = {mid} | get = {shown} > target -> hi = {mid - 1}")
            hi = mid - 1
    print("hết khoảng -> -1 | tổng số lần gọi get:", reader.calls)
    return -1

print("kết quả:", trace_doubling([-1, 0, 3, 5, 9, 12], 9))
print()
print("kết quả:", trace_doubling([-1, 0, 3, 5, 9, 12], 2))

secret (ẩn) = [-1, 0, 3, 5, 9, 12], target = 9
[nhân đôi] r =   1 | get(1) =   0 | < target -> r *= 2
[nhân đôi] r =   2 | get(2) =   3 | < target -> r *= 2
[nhân đôi] r =   4 | get(4) =   9 | >= target -> dừng
[nhị phân] khoảng [2, 4]
[nhị phân] lo = 2, hi = 4, mid = 3 | get = 5 < target -> lo = 4
[nhị phân] lo = 4, hi = 4, mid = 4 | get = 9 == target -> TÌM THẤY
tổng số lần gọi get: 5
kết quả: 4

secret (ẩn) = [-1, 0, 3, 5, 9, 12], target = 2
[nhân đôi] r =   1 | get(1) =   0 | < target -> r *= 2
[nhân đôi] r =   2 | get(2) =   3 | >= target -> dừng
[nhị phân] khoảng [1, 2]
[nhị phân] lo = 1, hi = 2, mid = 1 | get = 0 < target -> lo = 2
[nhị phân] lo = 2, hi = 2, mid = 2 | get = 3 > target -> hi = 1
hết khoảng -> -1 | tổng số lần gọi get: 4
kết quả: -1


Quan sát: với `target = 9`, nhân đôi dừng ở `r = 4` (`get(4) = 9`); phần tìm nhị phân trong `[2, 4]` cần tối đa 2 lần hỏi nữa. Với `target = 2` không có, khoảng thu về rỗng và trả `-1`. Hãy chú ý lần gọi `get(r)` ở pha nhân đôi **được tính** vào tổng số lần gọi.

### 8.2. Trace trường hợp `target` lớn hơn mọi phần tử (nhân đôi vượt biên)

Khi `target = 100 > 12`, `get(r)` cho tới lúc vượt biên mới trả `INF >= target`. Đây là chỗ `INF` đóng vai trò chốt chặn:

In [19]:
print("kết quả:", trace_doubling([-1, 0, 3, 5, 9, 12], 100))

secret (ẩn) = [-1, 0, 3, 5, 9, 12], target = 100
[nhân đôi] r =   1 | get(1) =   0 | < target -> r *= 2
[nhân đôi] r =   2 | get(2) =   3 | < target -> r *= 2
[nhân đôi] r =   4 | get(4) =   9 | < target -> r *= 2
[nhân đôi] r =   8 | get(8) = INF | >= target -> dừng
[nhị phân] khoảng [4, 8]
[nhị phân] lo = 4, hi = 8, mid = 6 | get = INF > target -> hi = 5
[nhị phân] lo = 4, hi = 5, mid = 4 | get = 9 < target -> lo = 5
[nhị phân] lo = 5, hi = 5, mid = 5 | get = 12 < target -> lo = 6
hết khoảng -> -1 | tổng số lần gọi get: 7
kết quả: -1


Nhân đôi dừng ở `r = 8` vì `get(8) = INF` (ngoài biên mảng độ dài 6); đoạn tìm nhị phân `[4, 8]` cho các `mid` trong và ngoài biên; các lần `INF` đều đẩy ta sang trái, cuối cùng không tìm thấy `100`.

### 8.3. Trace cách 4 (tìm độ dài rồi tìm nhị phân)

In [20]:
def trace_find_length(secret):
    reader = ArrayReader(secret)
    r = 1
    while True:
        v = reader.get(r)
        print(f"[nhân đôi] r = {r:>3} | get = {'INF' if v == INF else v}")
        if v == INF:
            break
        r *= 2
    lo, hi = r // 2, r
    print(f"độ dài n nằm trong [{lo}, {hi}] (n = chỉ số đầu tiên trả INF)")
    while lo < hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        if v == INF:
            print(f"[nhị phân] [{lo}, {hi}] mid = {mid}: INF -> hi = {mid}")
            hi = mid
        else:
            print(f"[nhị phân] [{lo}, {hi}] mid = {mid}: {v} (hợp lệ) -> lo = {mid + 1}")
            lo = mid + 1
    print(f"=> n = {lo}, tổng số lần gọi get: {reader.calls}")
    return lo

for s in ([-1, 0, 3, 5, 9, 12], [7], list(range(0, 40, 2))):
    assert trace_find_length(s) == len(s)
    print()

[nhân đôi] r =   1 | get = 0
[nhân đôi] r =   2 | get = 3
[nhân đôi] r =   4 | get = 9
[nhân đôi] r =   8 | get = INF
độ dài n nằm trong [4, 8] (n = chỉ số đầu tiên trả INF)
[nhị phân] [4, 8] mid = 6: INF -> hi = 6
[nhị phân] [4, 6] mid = 5: 12 (hợp lệ) -> lo = 6
=> n = 6, tổng số lần gọi get: 6

[nhân đôi] r =   1 | get = INF
độ dài n nằm trong [0, 1] (n = chỉ số đầu tiên trả INF)
[nhị phân] [0, 1] mid = 0: 7 (hợp lệ) -> lo = 1
=> n = 1, tổng số lần gọi get: 2

[nhân đôi] r =   1 | get = 2
[nhân đôi] r =   2 | get = 4
[nhân đôi] r =   4 | get = 8
[nhân đôi] r =   8 | get = 16
[nhân đôi] r =  16 | get = 32
[nhân đôi] r =  32 | get = INF
độ dài n nằm trong [16, 32] (n = chỉ số đầu tiên trả INF)
[nhị phân] [16, 32] mid = 24: INF -> hi = 24
[nhị phân] [16, 24] mid = 20: INF -> hi = 20
[nhị phân] [16, 20] mid = 18: 36 (hợp lệ) -> lo = 19
[nhị phân] [19, 20] mid = 19: 38 (hợp lệ) -> lo = 20
=> n = 20, tổng số lần gọi get: 10



### 8.4. Trace cách 3 (cận cố định) — số lần hỏi gần như không đổi

In [21]:
def trace_fixed(secret, target, hi=10**4):
    reader = ArrayReader(secret)
    lo, steps = 0, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        steps += 1
        shown = "INF" if v == INF else v
        if v == target:
            print(f"bước {steps}: [{lo}, {hi}] mid = {mid} get = {shown} -> TÌM THẤY")
            return mid
        if v < target:
            print(f"bước {steps}: [{lo}, {hi}] mid = {mid} get = {shown} < target -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"bước {steps}: [{lo}, {hi}] mid = {mid} get = {shown} >= target -> hi = {mid - 1}")
            hi = mid - 1
    return -1

print("kết quả:", trace_fixed([-1, 0, 3, 5, 9, 12], 9))

bước 1: [0, 10000] mid = 5000 get = INF >= target -> hi = 4999
bước 2: [0, 4999] mid = 2499 get = INF >= target -> hi = 2498
bước 3: [0, 2498] mid = 1249 get = INF >= target -> hi = 1248
bước 4: [0, 1248] mid = 624 get = INF >= target -> hi = 623
bước 5: [0, 623] mid = 311 get = INF >= target -> hi = 310
bước 6: [0, 310] mid = 155 get = INF >= target -> hi = 154
bước 7: [0, 154] mid = 77 get = INF >= target -> hi = 76
bước 8: [0, 76] mid = 38 get = INF >= target -> hi = 37
bước 9: [0, 37] mid = 18 get = INF >= target -> hi = 17
bước 10: [0, 17] mid = 8 get = INF >= target -> hi = 7
bước 11: [0, 7] mid = 3 get = 5 < target -> lo = 4
bước 12: [4, 7] mid = 5 get = 12 >= target -> hi = 4
bước 13: [4, 4] mid = 4 get = 9 -> TÌM THẤY
kết quả: 4


Ở cách 3, phần đầu của vòng lặp hỏi các chỉ số rất lớn (`5000`, `2500`, ...) đều trả `INF` — "đẩy" ta sang trái, dần về phần mảng thật. Đó là nhờ `INF` luôn lớn hơn `target`.

## 9. Code nộp LeetCode

Chọn **cách 2 (nhân đôi rồi tìm nhị phân)**: không phụ thuộc vào ràng buộc `n <= 10^4`, số lần gọi `get` là `O(log n)` theo **độ dài thật** của mảng. Đây là code đúng class/method/signature (trên LeetCode `ArrayReader` do hệ thống cung cấp; ở đây ta tự dựng ở đầu notebook):

In [22]:
class Solution:
    def search(self, reader: "ArrayReader", target: int) -> int:
        r = 1
        while reader.get(r) < target:        # nhân đôi tới khi get(r) >= target (hoặc ngoài biên)
            r *= 2
        lo, hi = r // 2, r
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid)
            if v == target:
                return mid
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1


for secret, target, expected in examples:
    reader = ArrayReader(secret)
    got = Solution().search(reader, target)
    print(f"secret = {secret}, target = {target} -> {got} (kỳ vọng {expected}), số lần gọi get = {reader.calls}")
    assert got == expected

secret = [-1, 0, 3, 5, 9, 12], target = 9 -> 4 (kỳ vọng 4), số lần gọi get = 5
secret = [-1, 0, 3, 5, 9, 12], target = 2 -> -1 (kỳ vọng -1), số lần gọi get = 4


## 10. Vì sao thuật toán đúng

**Mô hình.** Gọi `n` là độ dài thật của `secret`. Định nghĩa dãy mở rộng `A[i] = secret[i]` với `i < n` và `A[i] = INF = 2^31 - 1` với `i >= n`. Vì `secret` tăng chặt và mọi phần tử `< INF` nên `A` **không giảm** (tăng chặt trên `[0, n - 1]` và hằng `INF` sau đó). `reader.get(i)` chính là `A[i]` cho `i >= 0`. Với `target < INF`, tìm `target` trong `secret` tương đương tìm chỉ số `k` với `A[k] == target` (chỉ có thể `k < n`).

**Cách 2.**
- Pha nhân đôi: vì `A` không giảm và `A[i] = INF >= target` với mọi `i >= n`, vòng `while A[r] < target: r *= 2` **kết thúc** (khi `r >= n` nhiều nhất) và lúc dừng `A[r] >= target`. Ngay trước bước nhân đôi cuối, `A[r // 2] < target` (với `r >= 2`).
- Giả sử `A[k] == target`. Nếu `r >= 2`: mọi `i <= r // 2` có `A[i] <= A[r // 2] < target` nên `k > r // 2`; nếu `r = 1` thì `k >= 0 = r // 2`. Mặt khác `A[r] >= target = A[k]` và `A` tăng chặt trên phần thật (phần `INF` thì lớn hơn `target`), nên không thể `k > r` (khi đó `A[k] > A[r] >= target`, mâu thuẫn). Vậy `k` nằm trong `[r // 2, r]`.
- Pha nhị phân: tìm nhị phân chuẩn trên đoạn `[lo, hi] = [r // 2, r]` của dãy không giảm `A` tìm giá trị `target` — đúng. Mỗi `get(mid)` ngoài biên trả `INF > target` nên đưa ta sang trái, đúng hướng.
- Nếu không thấy trong đoạn thì không tồn tại (vì mọi `k` có thể đều thuộc đoạn) → `-1`.

**Số lần gọi.** Pha nhân đôi: `r` chạy `1, 2, 4, ..., 2^j` với `2^j` là lũy thừa 2 nhỏ nhất `>= k` (khi tìm thấy) hoặc `>= n` (khi không): tối đa `ceil(log2(n)) + 1` lần gọi. Pha nhị phân trên đoạn có `r / 2 + 1` phần tử; vì `r` là lũy thừa 2 nhỏ nhất thỏa `A[r] >= target` nên `r < 2 * max(k, 1)` hoặc `r < 2n` (khi `target` vắng), tức đoạn dài không quá `n + 1`: tối đa `ceil(log2(n + 2))` lần gọi. Tổng `O(log n)`; ta sẽ **đo** và kiểm chứng ở mục 12.

**Cách 3.** Tìm nhị phân trên `[0, H]` của dãy không giảm `A` với `H >= n - 1`. Mọi `k` hợp lệ nằm trong `[0, n - 1] ⊆ [0, H]` nên tìm thấy nếu tồn tại.

**Cách 4.** Pha 1: tìm chỉ số đầu tiên có `A[i] == INF`, đó là `n` (vì `A[i] < INF` với `i < n`, và `= INF` với `i >= n`); nhân đôi chốt cận rồi tìm nhị phân điểm chuyển. Pha 2: tìm nhị phân thường trên `[0, n - 1]`.

**Cách 1.** Duyệt tăng chỉ số và dừng khi `A[i] >= target` (đúng vì không giảm): nếu `A[i] == target` trả `i`, ngược lại `-1`.

## 11. Time / Space Complexity

(`n` là độ dài thật của mảng; "độ phức tạp" ở đây là số lần gọi `get` — các phép tính khác chỉ `O(1)` mỗi lần.)

| Cách | Số lần gọi `get` | Space |
|---|---|---|
| 1. Duyệt tuyến tính | `O(n)` | `O(1)` |
| 2. Nhân đôi + nhị phân (nộp) | `O(log n)` (≈ `2 log2 k`) | `O(1)` |
| 3. Cận cố định `H` + nhị phân | `O(log H)` (~14 với `H = 10^4`; ~31 với `H = 2^31 - 1`) | `O(1)` |
| 4. Tìm độ dài rồi nhị phân | `O(log n)` (≈ `3 log2 n`) | `O(1)` |

Ta đo số lần gọi thực tế trên nhiều kích thước:

In [23]:
def calls_for(solver, secret, target):
    reader = ArrayReader(secret)
    ans = solver(reader, target)
    return ans, reader.calls


print(f"{'n':>6} | {'tuyến tính':>10} | {'nhân đôi':>9} | {'cận 10^4':>9} | {'tìm độ dài':>10} | 2*log2(n)+3")
for n in (1, 2, 10, 100, 1000, 5000, 10000):
    secret = list(range(0, 2 * n, 2))
    target = secret[-1]                      # phần tử cuối: trường hợp xấu cho nhiều cách
    row = []
    for solver in (search_linear, Solution2().search, Solution3().search, Solution4().search):
        ans, calls = calls_for(solver, secret, target)
        assert ans == n - 1
        row.append(calls)
    print(f"{n:>6} | {row[0]:>10} | {row[1]:>9} | {row[2]:>9} | {row[3]:>10} | {2 * math.log2(n) + 3:6.1f}")

     n | tuyến tính |  nhân đôi |  cận 10^4 | tìm độ dài | 2*log2(n)+3
     1 |          1 |         2 |        13 |          3 |    3.0
     2 |          2 |         3 |        12 |          5 |    5.0
    10 |         10 |         7 |        13 |         12 |    9.6
   100 |        100 |        14 |        13 |         21 |   16.3
  1000 |       1000 |        20 |        11 |         30 |   22.9
  5000 |       5000 |        23 |        14 |         39 |   27.6
 10000 |      10000 |        24 |        13 |         42 |   29.6


Cách 2 chỉ gọi `get` cỡ `2 * log2 n` lần, còn duyệt tuyến tính gọi `n + 1` lần. Cách 3 gọi gần như hằng số (~14) dù `n` bé hay lớn, vì cận `10^4` cố định.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta viết `check_all` chạy mọi cách trên cùng `(secret, target)` bằng một `ArrayReader` **mới cho mỗi lời gọi** (để bộ đếm không lẫn) và đòi tất cả trùng với đáp án đúng (tính độc lập bằng `list.index`).

In [24]:
def oracle(secret, target):
    """Oracle độc lập: dùng index của list (không dùng ArrayReader)."""
    return secret.index(target) if target in secret else -1


solvers = [
    ("tuyến tính", search_linear),
    ("cách 2", Solution2().search),
    ("cách 3", Solution3().search),
    ("cách 4", Solution4().search),
    ("bisect", search_bisect),
]


def check_all(secret, target, expected=None):
    want = oracle(secret, target)
    if expected is not None:
        assert want == expected, (secret, target, want, expected)
    for name, f in solvers:
        reader = ArrayReader(secret)
        got = f(reader, target)
        assert got == want, (name, secret, target, got, want)
    return want


for secret, target, expected in examples:
    check_all(secret, target, expected)
print("ví dụ đề: OK")

edge = [
    ([5], 5, 0),                 # một phần tử, có
    ([5], 4, -1),                # một phần tử, nhỏ hơn
    ([5], 6, -1),                # một phần tử, lớn hơn
    ([1, 2], 1, 0),
    ([1, 2], 2, 1),
    ([1, 2], 3, -1),
    ([-10**4, 10**4], -10**4, 0),            # biên giá trị
    ([-10**4, 10**4], 10**4, 1),
    ([-10**4, 0, 10**4], 0, 1),
    ([-1, 0, 3, 5, 9, 12], -1, 0),           # phần tử đầu
    ([-1, 0, 3, 5, 9, 12], 12, 5),           # phần tử cuối
    ([-1, 0, 3, 5, 9, 12], -5, -1),          # nhỏ hơn tất cả
    ([-1, 0, 3, 5, 9, 12], 100, -1),         # lớn hơn tất cả
    (list(range(0, 8)), 7, 7),               # độ dài đúng lũy thừa 2 (8)
    (list(range(0, 9)), 8, 8),               # lũy thừa 2 cộng 1
    (list(range(0, 7)), 6, 6),               # lũy thừa 2 trừ 1
    (list(range(0, 16)), 15, 15),
    (list(range(0, 16)), 8, 8),              # đúng điểm r của nhân đôi
]
for secret, target, expected in edge:
    check_all(secret, target, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


### 12.2. Số lần gọi `get` phải là `O(log n)`

Bài tương tác có yêu cầu về độ phức tạp, nên test **không chỉ đúng đáp án mà còn đúng số lần gọi**. Với cách 2, ta chứng minh ở mục 10 rằng số lần gọi `<= 2 * ceil(log2(n + 1)) + 4` (độ rộng nới thêm vài đơn vị cho các trường hợp biên). Ta kiểm tra bằng `assert` trên **mọi** `n` nhỏ và `target` và trên nhiều `n` lớn.

In [25]:
def bound(n):
    return 2 * math.ceil(math.log2(n + 1)) + 4


worst = {}
for n in range(1, 130):
    secret = list(range(0, 2 * n, 2))
    targets = secret + [t for t in range(-3, 2 * n + 3) if t not in secret]    # mọi phần tử và nhiều giá trị vắng
    for t in targets:
        reader = ArrayReader(secret)
        ans = Solution2().search(reader, t)
        assert ans == oracle(secret, t)
        assert reader.calls <= bound(n), (n, t, reader.calls, bound(n))
        worst[n] = max(worst.get(n, 0), reader.calls)
print("mọi n = 1..129, mọi target: cách 2 đúng và số lần gọi <= 2*ceil(log2(n+1)) + 4")
print("số lần gọi tối đa quan sát được với n = 1, 2, 3, 8, 64, 128:", [worst[n] for n in (1, 2, 3, 8, 64, 128)])

rng = random.Random(7)
for _ in range(200):
    n = rng.randint(130, 10**4)
    secret = sorted(rng.sample(range(-10**4, 10**4 + 1), n))
    for t in [rng.choice(secret), secret[0], secret[-1], rng.randint(-10**4, 10**4)]:
        reader = ArrayReader(secret)
        assert Solution2().search(reader, t) == oracle(secret, t)
        assert reader.calls <= bound(n), (n, t, reader.calls)
print("200 mảng lớn ngẫu nhiên (n tới 10^4): đúng và số lần gọi <= bound")

mọi n = 1..129, mọi target: cách 2 đúng và số lần gọi <= 2*ceil(log2(n+1)) + 4


số lần gọi tối đa quan sát được với n = 1, 2, 3, 8, 64, 128: [3, 4, 5, 7, 13, 15]


200 mảng lớn ngẫu nhiên (n tới 10^4): đúng và số lần gọi <= bound


### 12.3. Phản ví dụ: duyệt tuyến tính vi phạm `O(log n)`

Cách 1 trả đúng đáp án nhưng số lần gọi tăng tuyến tính, nên **bị loại** về yêu cầu độ phức tạp:

In [26]:
secret = list(range(0, 20000, 2))     # n = 10^4
target = secret[-1]
for name, f in (("tuyến tính", search_linear), ("cách 2", Solution2().search)):
    ans, calls = calls_for(f, secret, target)
    print(f"{name:11}: đáp án {ans}, số lần gọi get = {calls}")
assert calls_for(search_linear, secret, target)[1] > bound(len(secret))

tuyến tính : đáp án 9999, số lần gọi get = 10000
cách 2     : đáp án 9999, số lần gọi get = 24


### 12.4. Đối chiếu ngẫu nhiên giữa các cách (seed cố định)

Sinh mảng **đúng ràng buộc**: tăng chặt, độ dài `1..10^4`, giá trị trong `[-10^4, 10^4]`, và `target` trong cùng khoảng (cả giá trị có và không có). So mọi cách (kể cả `bisect`) với oracle.

In [27]:
rng = random.Random(20702)
total = found = 0
for _ in range(3000):
    n = rng.randint(1, 40)
    secret = sorted(rng.sample(range(-60, 61), n))       # dải nhỏ -> target hay trùng
    t = rng.randint(-65, 65)
    want = check_all(secret, t)
    total += 1
    found += want != -1
print(f"{total} mảng nhỏ ngẫu nhiên: mọi cách khớp oracle ({found} có target, {total - found} không có)")

for _ in range(150):
    n = rng.randint(1, 10**4)
    secret = sorted(rng.sample(range(-10**4, 10**4 + 1), n))
    for t in (rng.choice(secret), rng.randint(-10**4, 10**4)):
        want = oracle(secret, t)
        for name, f in solvers[1:]:                      # bỏ tuyến tính cho nhanh
            assert f(ArrayReader(secret), t) == want, (name, n, t)
print("150 mảng lớn: các cách nhị phân khớp oracle")

# mảng đầy đủ nhất: 10^4 phần tử, target tại mọi vị trí ở các mốc đặc biệt
secret = list(range(-5000, 5000))
for idx in (0, 1, 2, 3, 4, 7, 8, 9, 15, 16, 17, 1023, 1024, 1025, 4095, 4096, 4097, 9998, 9999):
    assert Solution().search(ArrayReader(secret), secret[idx]) == idx
print("mảng 10^4 phần tử: các mốc lũy thừa 2 OK")

3000 mảng nhỏ ngẫu nhiên: mọi cách khớp oracle (489 có target, 2511 không có)


150 mảng lớn: các cách nhị phân khớp oracle
mảng 10^4 phần tử: các mốc lũy thừa 2 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhân đôi bắt đầu từ `r = 0`

**Code sai:**

In [28]:
class Wrong1:
    def search(self, reader, target):
        r = 0
        guard = 0
        while reader.get(r) < target:
            r *= 2                       # SAI: 0 * 2 = 0, không bao giờ tăng
            guard += 1
            if guard > 50:               # chỉ để notebook không bị treo
                raise RuntimeError("lặp vô hạn: r mãi bằng 0")
        return r

try:
    Wrong1().search(ArrayReader([-1, 0, 3, 5, 9, 12]), 9)
except RuntimeError as ex:
    print("lỗi:", ex)
print("đúng:", Solution().search(ArrayReader([-1, 0, 3, 5, 9, 12]), 9))

lỗi: lặp vô hạn: r mãi bằng 0
đúng: 4


**Vì sao sai:** `0 * 2 = 0`, nên `r` mãi bằng `0` và nếu `secret[0] < target` vòng `while` không bao giờ kết thúc. Phải bắt đầu từ `r = 1`. (Nhớ rằng khi `r = 1`, đoạn tìm nhị phân là `[0, 1]`, vì ta chưa kiểm tra `get(0)`.)

### Lỗi 2: Đặt `hi = r - 1` làm mất phần tử ở chỗ dừng

In [29]:
class Wrong2:
    def search(self, reader, target):
        r = 1
        while reader.get(r) < target:
            r *= 2
        lo, hi = r // 2, r - 1           # SAI: bỏ chỉ số r, nhưng get(r) có thể đúng bằng target
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid)
            if v == target:
                return mid
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1

secret = [-1, 0, 3, 5, 9, 12]
for t in (9, 3, 12, -1):
    print(f"target = {t}: sai -> {Wrong2().search(ArrayReader(secret), t)} | đúng -> {Solution().search(ArrayReader(secret), t)}")

target = 9: sai -> -1 | đúng -> 4
target = 3: sai -> -1 | đúng -> 2
target = 12: sai -> 5 | đúng -> 5
target = -1: sai -> 0 | đúng -> 0


**Vì sao sai:** vòng nhân đôi dừng khi `get(r) >= target`; nếu `get(r) == target` thì phần tử nằm **đúng tại `r`**. Đặt `hi = r - 1` bỏ qua nó. Với `target = 9` và `r = 4` (`get(4) = 9`) kết quả sai. Cận đúng là `hi = r`.

### Lỗi 3: Duyệt tuyến tính — đúng nhưng vi phạm `O(log n)`
Đã thấy ở mục 12.3: cách 1 trả đúng đáp án nhưng gọi `get` `O(n)` lần, bị hệ chấm tương tác coi là không đạt (số lần gọi vượt giới hạn). **Với bài tương tác, độ phức tạp theo số lần gọi cũng là một phần của đề.**

In [30]:
secret = list(range(0, 20000, 2))
print("duyệt tuyến tính:", calls_for(search_linear, secret, secret[-1]))
print("nhân đôi + nhị phân:", calls_for(Solution().search, secret, secret[-1]))

duyệt tuyến tính: (9999, 10000)
nhân đôi + nhị phân: (9999, 24)


### Lỗi 4: Tưởng `reader` là mảng thường (`len(reader)`, `reader[i]`)

In [31]:
class Wrong4:
    def search(self, reader, target):
        lo, hi = 0, len(reader) - 1            # SAI: reader không có len
        while lo <= hi:
            mid = (lo + hi) // 2
            if reader[mid] == target:           # SAI: reader không hỗ trợ []
                return mid
            lo = mid + 1
        return -1

try:
    Wrong4().search(ArrayReader([1, 2, 3]), 2)
except TypeError as ex:
    print("TypeError:", ex)

TypeError: object of type 'ArrayReader' has no len()


**Vì sao sai:** `ArrayReader` chỉ có phương thức `get(i)`. Bài toán chính là **không biết độ dài**, nên không dùng được `len` hay toán tử `[]`. Mọi truy cập phải qua `reader.get(i)`.

### Lỗi 5: Hỏi chỉ số âm

In [32]:
class Wrong5:
    def search(self, reader, target):
        lo, hi = 0, 10**4
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid - 1)            # SAI: lùi một chỉ số, có thể thành -1
            if v == target:
                return mid - 1
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1

try:
    Wrong5().search(ArrayReader([5, 6, 7]), -100)
    print("không lỗi")
except IndexError as ex:
    print("IndexError:", ex)

IndexError: chỉ số âm không hợp lệ: -1


**Vì sao sai:** chỉ số hợp lệ bắt đầu từ `0`. Đề chỉ nói `get(i)` trả `INF` khi `i` **vượt phía phải**; chỉ số âm **không được định nghĩa** (hệ chấm có thể báo lỗi). Luôn giữ `lo >= 0`. Trong lớp `ArrayReader` của notebook ta cố ý báo `IndexError` để bắt loại lỗi này.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mảng đã sắp xếp nhưng **không biết độ dài** / 'vô hạn' / chỉ truy cập qua API" → **exponential search (galloping)**: nhân đôi cận rồi tìm nhị phân trong khoảng vừa tìm được. Tổng `O(log k)` với `k` là vị trí thật.
- **Giá trị "chốt chặn" (sentinel):** thứ trả về khi ngoài biên (`2^31 - 1`) lớn hơn mọi dữ liệu thật ⇒ giữ dãy **đơn điệu**, cho phép tìm nhị phân tự nhiên. Nhận ra sentinel đơn điệu là chìa khóa.
- **Hai pha:** (1) tìm cận, (2) tìm nhị phân trong cận. Mẫu này còn gặp ở: tìm điểm đầu tiên thỏa điều kiện (`First Bad Version`), tìm biên trong dãy rất dài, `Find in Mountain Array`.
- **Bài tương tác:** **đếm số lần gọi API** vì đề thường giới hạn. Hãy tự dựng lớp giả (mock) có bộ đếm để kiểm thử.
- **Bắt đầu nhân đôi từ `1`**, không phải `0`; và nhớ khoảng tìm nhị phân là `[r // 2, r]` **bao gồm `r`**.
- **Cận cố định** (nếu đề cho `n <= 10^4`) là một lối tắt hợp lệ — nhưng chỉ khi tin vào ràng buộc; nhân đôi tổng quát hơn.
- **Python:** `bisect` làm việc trên mọi đối tượng có `__len__` và `__getitem__`; `r *= 2` hoặc `r <<= 1` đều được.
- **Test:** vừa kiểm tra **đúng đáp án** vừa kiểm tra **số lần gọi** (hằng số nhỏ, bậc `log n`).

## 15. Biến thể

1. **Tìm vị trí chèn (lower bound) trong mảng kích thước không biết:** chỉ số đầu tiên có giá trị `>= target` (xem demo).
2. **Tìm độ dài mảng** (`find_length` ở cách 4) — đáp án chính là điểm chuyển sang `INF`.
3. **Mảng có phần tử trùng:** tìm chỉ số **đầu tiên** hoặc **cuối cùng** của `target`.
4. **`Find in Mountain Array` (LeetCode 1095):** mảng tăng rồi giảm, cũng truy cập qua API có giới hạn số lần gọi.
5. **`First Bad Version` (LeetCode 278)** và **`Binary Search` (LeetCode 704):** nền tảng của pha thứ hai.
6. **Luồng/dữ liệu vô hạn:** tìm trong một dãy tăng vô hạn bằng galloping (giá trị lớn dần).
7. **Search Insert Position (LeetCode 35):** cùng bài lower bound nhưng biết độ dài.

Demo biến thể 1 và 3 (đối chiếu với `bisect`):

In [33]:
def lower_bound_unknown(reader, target):
    """Chỉ số đầu tiên có get(i) >= target (n nếu mọi phần tử thật < target). Mảng có thể có phần tử trùng."""
    r = 1
    while reader.get(r) < target:
        r *= 2
    lo, hi = r // 2, r
    while lo < hi:
        mid = (lo + hi) // 2
        if reader.get(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo


def first_occurrence(reader, target):
    i = lower_bound_unknown(reader, target)
    return i if reader.get(i) == target else -1


rng = random.Random(15)
for _ in range(2000):
    n = rng.randint(1, 30)
    secret = sorted(rng.choices(range(-12, 13), k=n))      # có thể trùng
    t = rng.randint(-14, 14)
    reader = ArrayReader(secret)
    assert lower_bound_unknown(reader, t) == bisect_left(secret, t), (secret, t)
    want = secret.index(t) if t in secret else -1
    assert first_occurrence(ArrayReader(secret), t) == want
print("lower bound và vị trí xuất hiện đầu tiên: khớp bisect/index trên 2000 mảng có phần tử trùng")
print(lower_bound_unknown(ArrayReader([1, 3, 5, 7]), 4), lower_bound_unknown(ArrayReader([1, 3, 5, 7]), 100))

lower bound và vị trí xuất hiện đầu tiên: khớp bisect/index trên 2000 mảng có phần tử trùng
2 4


## 16. So sánh các cách

| Cách | Ý tưởng | Số lần gọi `get` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tuyến tính | Hỏi lần lượt `0, 1, 2, ...` | `O(n)` | `O(1)` | Chỉ để kiểm chứng; **không đạt** yêu cầu |
| 2. Nhân đôi + nhị phân | Chốt cận `r` rồi tìm nhị phân trong `[r/2, r]` | `O(log n)` (≈ `2 log2 k`) | `O(1)` | **Cách nộp**, tổng quát |
| 3. Cận cố định | Nhị phân trên `[0, H]` nhờ sentinel `INF` | `O(log H)` | `O(1)` | Khi tin vào ràng buộc `n <= H`; ngắn nhất |
| 4. Tìm độ dài rồi nhị phân | Hai bài nhị phân độc lập | `O(log n)` (≈ `3 log2 n`) | `O(1)` | Khi cần biết độ dài, hoặc muốn tách bài toán |

Khác nhau thật về ý tưởng: cách 1 không dùng thứ tự để loại nửa; cách 2 **chốt cận bằng cách nhân đôi**; cách 3 **đoán cận lớn** dựa trên sentinel; cách 4 **tìm kích thước** trước.

In [34]:
# so sánh số lần gọi get trung bình trên nhiều mảng ngẫu nhiên (tìm các giá trị có trong mảng)
rng = random.Random(16)
stats = {name: [] for name, _ in solvers[:4]}
for _ in range(300):
    n = rng.randint(1, 10**4)
    secret = sorted(rng.sample(range(-10**4, 10**4 + 1), n))
    t = rng.choice(secret)
    for name, f in solvers[:4]:
        if name == "tuyến tính" and n > 3000:
            continue
        reader = ArrayReader(secret)
        assert f(reader, t) == secret.index(t)
        stats[name].append(reader.calls)
for name, vals in stats.items():
    print(f"{name:11}: trung bình {sum(vals) / len(vals):8.1f} lần gọi, tối đa {max(vals):>6}")

tuyến tính : trung bình    790.3 lần gọi, tối đa   2727
cách 2     : trung bình     20.6 lần gọi, tối đa     28
cách 3     : trung bình     12.3 lần gọi, tối đa     14
cách 4     : trung bình     35.7 lần gọi, tối đa     42


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [35]:
r = ArrayReader([2, 4, 6, 8, 10])
print(Solution().search(r, 8), "| số lần gọi:", r.calls)
print(Solution().search(ArrayReader([2, 4, 6, 8, 10]), 5))
print(Solution().search(ArrayReader([7]), 7))

3 | số lần gọi: 4
-1
0


**Bài 2 (tính tay).** Với `secret = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]` và `target = 13`, trong cách 2: (a) các giá trị `r` được thử ở pha nhân đôi (liệt kê theo thứ tự, gồm cả `r` cuối), gán list vào `ans2a`; (b) khoảng `[lo, hi]` cho pha nhị phân, gán tuple `(lo, hi)` vào `ans2b`; (c) chỉ số trả về, gán vào `ans2c` (đều đang là `None`).

In [36]:
ans2a = None   # TODO: list các r đã hỏi
ans2b = None   # TODO: tuple (lo, hi)
ans2c = None   # TODO: chỉ số trả về

**Bài 3 (điền TODO).** Hoàn thành `find_size(reader)` trả về **độ dài** của mảng bí mật (số phần tử thật), bằng nhân đôi + nhị phân điểm chuyển, dùng `O(log n)` lần gọi.

In [37]:
def find_size(reader):
    # TODO: nhân đôi r tới khi get(r) == INF, rồi tìm nhị phân chỉ số đầu tiên trả INF trong [r // 2, r]
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_search(reader, target)` theo cách 2.

In [38]:
def my_search(reader, target):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `count_in_range(reader, lo_value, hi_value)`: số phần tử `x` của mảng bí mật với `lo_value <= x <= hi_value`, dùng `O(log n)` lần gọi (gợi ý: hai lần lower bound). (b) Viết `search_nonstrict(reader, target)`: như bài gốc nhưng mảng **không giảm** (có thể có phần tử trùng); trả về chỉ số **đầu tiên** của `target` hoặc `-1`.

In [39]:
def count_in_range(reader, lo_value, hi_value):
    # TODO
    pass

def search_nonstrict(reader, target):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: thử từng bước như mục 4: hỏi `get(1), get(2), get(4)`, ...
- Bài 2: bắt đầu `r = 1`, nhân đôi tới khi `get(r) >= 13`.
- Bài 3: chỉ số đầu tiên trả `INF` chính là độ dài.
- Bài 4: nhân đôi `r` từ 1, rồi tìm nhị phân trong `[r // 2, r]`.
- Bài 5: số phần tử trong đoạn `[a, b]` = (số phần tử `< b + 1`) − (số phần tử `< a`).

**Mức 2 (vừa):**
- Bài 2: `get(1) = 3`, `get(2) = 5`, `get(4) = 9`, `get(8) = 17 >= 13` ⇒ `r = 8`; đoạn `[4, 8]`.
- Bài 3: `while reader.get(r) != INF: r *= 2`; rồi `lo, hi = r // 2, r`; `while lo < hi: mid...; if get(mid) == INF: hi = mid else lo = mid + 1`.
- Bài 4: nhớ `hi = r` (không phải `r - 1`).
- Bài 5: (a) viết `lower_bound(value)` = chỉ số đầu tiên có `get >= value`; đáp án `lower_bound(hi_value + 1) - lower_bound(lo_value)`. (b) `i = lower_bound(target)` rồi kiểm tra `get(i) == target`.

**Mức 3 (gần đáp án):**
- Bài 2: pha nhị phân: `mid = 6`, `get(6) = 13` → tìm thấy.
- Bài 3: xem `find_length` ở mục 7.3.
- Bài 4: xem mục 9, code nộp.
- Bài 5: xem `lower_bound_unknown` ở mục 15.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [40]:
# Bài 1
r = ArrayReader([2, 4, 6, 8, 10])
assert Solution().search(r, 8) == 3
assert r.calls <= bound(5)
assert Solution().search(ArrayReader([2, 4, 6, 8, 10]), 5) == -1
assert Solution().search(ArrayReader([7]), 7) == 0

# Bài 2
ans2a = [1, 2, 4, 8]
ans2b = (4, 8)
ans2c = 6
secret2 = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
rr = ArrayReader(secret2)
asked = []
x = 1
while rr.get(x) < 13:
    asked.append(x)
    x *= 2
asked.append(x)
assert ans2a == asked
assert ans2b == (x // 2, x)
assert ans2c == Solution().search(ArrayReader(secret2), 13) == secret2.index(13)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [41]:
# Bài 3
def find_size(reader):
    r = 1
    while reader.get(r) != INF:
        r *= 2
    lo, hi = r // 2, r
    while lo < hi:
        mid = (lo + hi) // 2
        if reader.get(mid) == INF:
            hi = mid
        else:
            lo = mid + 1
    return lo

for n in range(1, 200):
    reader = ArrayReader(list(range(n)))
    assert find_size(reader) == n, n
    assert reader.calls <= 2 * math.ceil(math.log2(n + 1)) + 4
reader = ArrayReader(list(range(10**4)))
assert find_size(reader) == 10**4
print("Bài 3 OK")

Bài 3 OK


In [42]:
# Bài 4
def my_search(reader, target):
    r = 1
    while reader.get(r) < target:
        r *= 2
    lo, hi = r // 2, r
    while lo <= hi:
        mid = (lo + hi) // 2
        v = reader.get(mid)
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 60)
    secret = sorted(rng.sample(range(-80, 81), n))
    t = rng.randint(-85, 85)
    assert my_search(ArrayReader(secret), t) == oracle(secret, t)
print("Bài 4 OK")

Bài 4 OK


In [43]:
# Bài 5
def count_in_range(reader, lo_value, hi_value):
    def lower_bound(value):
        r = 1
        while reader.get(r) < value:
            r *= 2
        lo, hi = r // 2, r
        while lo < hi:
            mid = (lo + hi) // 2
            if reader.get(mid) >= value:
                hi = mid
            else:
                lo = mid + 1
        return lo
    if lo_value > hi_value:
        return 0
    return lower_bound(hi_value + 1) - lower_bound(lo_value)


def search_nonstrict(reader, target):
    return first_occurrence(reader, target)


rng = random.Random(5)
for _ in range(1000):
    n = rng.randint(1, 40)
    secret = sorted(rng.choices(range(-15, 16), k=n))
    a, b = sorted((rng.randint(-18, 18), rng.randint(-18, 18)))
    want = sum(1 for x in secret if a <= x <= b)
    assert count_in_range(ArrayReader(secret), a, b) == want, (secret, a, b)
    t = rng.randint(-18, 18)
    want_first = secret.index(t) if t in secret else -1
    assert search_nonstrict(ArrayReader(secret), t) == want_first
assert count_in_range(ArrayReader([1, 3, 5, 7]), 4, 3) == 0
reader = ArrayReader(list(range(10**4)))
assert count_in_range(reader, 100, 9000) == 8901
assert reader.calls <= 4 * (math.ceil(math.log2(10**4)) + 3)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template nhân đôi + nhị phân (nộp):**

```python
class Solution:
    def search(self, reader: 'ArrayReader', target: int) -> int:
        r = 1
        while reader.get(r) < target:      # nhân đôi tới khi get(r) >= target (hoặc ngoài biên = 2^31 - 1)
            r *= 2
        lo, hi = r // 2, r                 # target (nếu có) nằm trong [r//2, r]
        while lo <= hi:
            mid = (lo + hi) // 2
            v = reader.get(mid)
            if v == target:
                return mid
            if v < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1
```

**Template cận cố định:**

```python
lo, hi = 0, 10**4
while lo <= hi:
    mid = (lo + hi) // 2
    v = reader.get(mid)
    if v == target: return mid
    if v < target: lo = mid + 1
    else: hi = mid - 1
return -1
```

**Checklist trước khi submit:**

- [ ] Nhân đôi bắt đầu từ `r = 1` (không phải `0`)?
- [ ] Khoảng nhị phân là `[r // 2, r]` và `hi = r` (**bao gồm** `r`)?
- [ ] Chỉ truy cập mảng qua `reader.get(i)` (không `len`, không `[]`, không chỉ số âm)?
- [ ] Hiểu rằng `2^31 - 1` (ngoài biên) lớn hơn `target` nên coi như `+vô cùng`?
- [ ] Số lần gọi `get` là `O(log n)` (không duyệt tuyến tính)?
- [ ] Đã test: một phần tử, `target` nhỏ hơn/lớn hơn tất cả, phần tử đầu/cuối, độ dài là lũy thừa 2 (±1)?
- [ ] Đã so với oracle `list.index` và kiểm tra bộ đếm số lần gọi?